# 🧠 Sentiment Analysis using NLP Pipeline & ML Models
**Assignment NLP-2 | Data Science Internship – February 2026**

---

## Objective
Build a complete end-to-end Sentiment Analysis system by applying:
- NLP Preprocessing
- Feature Engineering (BoW & TF-IDF)
- Multiple ML Models (Logistic Regression, Naive Bayes, Decision Tree, Random Forest, XGBoost)
- Model Evaluation & Comparison

**Dataset:** IMDb Movie Reviews Dataset (from Kaggle)  
**Labels:** Positive / Negative

---
### Pipeline Flow
```
Raw Data → Preprocessing → Feature Engineering → Model Training → Evaluation → Comparison
```

## 📦 Step 0: Install Required Libraries

In [ ]:
# Install required libraries (run once in Colab/Jupyter)
!pip install nltk scikit-learn pandas numpy matplotlib seaborn xgboost wordcloud --quiet

## 📚 Step 1: Import Libraries

In [ ]:
# ── Standard Libraries ──────────────────────────────────────────────────────
import re
import string
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ── Visualization ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud

# ── NLP ──────────────────────────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download required NLTK data
nltk.download('punkt',        quiet=True)
nltk.download('punkt_tab',   quiet=True)
nltk.download('stopwords',   quiet=True)
nltk.download('wordnet',     quiet=True)
nltk.download('omw-1.4',     quiet=True)

# ── Feature Engineering ──────────────────────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# ── ML Models ────────────────────────────────────────────────────────────────
from sklearn.linear_model    import LogisticRegression
from sklearn.naive_bayes     import MultinomialNB
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier

# ── Model Selection & Evaluation ─────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics         import (accuracy_score, precision_score,
                                     recall_score, f1_score,
                                     classification_report, confusion_matrix)

print("✅ All libraries imported successfully!")

---
## 📂 Step 2: Data Loading & Understanding

We use the **IMDb Movie Reviews** dataset from Kaggle.  
It contains 50,000 reviews labelled as `positive` or `negative`.

> **How to get the dataset:**  
> 1. Go to https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews  
> 2. Download `IMDB Dataset.csv`  
> 3. Upload it to your Colab session or place it in the same folder as this notebook.

In [ ]:
# ── Load the dataset ─────────────────────────────────────────────────────────
# Update the path if your file is in a different location
df = pd.read_csv('IMDB Dataset.csv')

print("Dataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# ── Basic Exploration ────────────────────────────────────────────────────────
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total Samples     : {len(df)}")
print(f"Missing Values    : {df.isnull().sum().sum()}")
print(f"Duplicate Rows    : {df.duplicated().sum()}")
print("\nClass Distribution:")
print(df['sentiment'].value_counts())
print("\nSample Review (Positive):")
print(df[df['sentiment'] == 'positive']['review'].iloc[0][:300], '...')
print("\nSample Review (Negative):")
print(df[df['sentiment'] == 'negative']['review'].iloc[0][:300], '...')

In [ ]:
# ── Visualize Class Distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar Chart
sentiment_counts = df['sentiment'].value_counts()
axes[0].bar(sentiment_counts.index, sentiment_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.4)
axes[0].set_title('Class Distribution (Bar Chart)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
for i, v in enumerate(sentiment_counts.values):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

# Pie Chart
axes[1].pie(sentiment_counts.values, labels=sentiment_counts.index,
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution (Pie Chart)', fontsize=13, fontweight='bold')

plt.suptitle('IMDb Sentiment Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Review length distribution
df['review_length'] = df['review'].apply(len)
plt.figure(figsize=(10, 4))
df.groupby('sentiment')['review_length'].plot(kind='hist', bins=50, alpha=0.6,
                                               color=['#2ecc71', '#e74c3c'], legend=True)
plt.title('Review Length Distribution by Sentiment', fontsize=13, fontweight='bold')
plt.xlabel('Review Length (characters)')
plt.ylabel('Frequency')
plt.legend(['Positive', 'Negative'])
plt.tight_layout()
plt.show()

---
## 🔧 Step 3: NLP Preprocessing (Mandatory)

We apply the following preprocessing steps in order:

| Step | Description |
|------|-------------|
| 1 | Remove HTML tags |
| 2 | Remove URLs |
| 3 | Lowercase conversion |
| 4 | Remove punctuation & special characters |
| 5 | Tokenization |
| 6 | Remove stopwords |
| 7 | Lemmatization |

All steps are wrapped in **reusable functions** for clean, modular code.

In [ ]:
# ── Initialise NLP Tools ─────────────────────────────────────────────────────
lemmatizer = WordNetLemmatizer()
stemmer    = PorterStemmer()
STOP_WORDS = set(stopwords.words('english'))

# ── Reusable Preprocessing Functions ─────────────────────────────────────────

def remove_html_tags(text):
    """Remove HTML tags like <br />, <p>, etc."""
    clean = re.compile('<.*?>')
    return re.sub(clean, ' ', text)

def remove_urls(text):
    """Remove URLs starting with http, https, www."""
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

def to_lowercase(text):
    """Convert all characters to lowercase."""
    return text.lower()

def remove_punctuation_special(text):
    """Remove punctuation and special characters, keep only letters and spaces."""
    text = re.sub(r'[^a-z\s]', '', text)   # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()  # collapse multiple spaces
    return text

def tokenize(text):
    """Split text into list of word tokens."""
    return word_tokenize(text)

def remove_stopwords(tokens):
    """Remove common English stopwords from token list."""
    return [word for word in tokens if word not in STOP_WORDS]

def lemmatize_tokens(tokens):
    """Lemmatize: reduce words to their dictionary base form."""
    return [lemmatizer.lemmatize(token) for token in tokens]

def stem_tokens(tokens):
    """Stemming: aggressively reduce words to their root stem."""
    return [stemmer.stem(token) for token in tokens]

def full_preprocess(text, use_stemming=False):
    """
    Complete NLP preprocessing pipeline.
    Args:
        text (str): raw input review text
        use_stemming (bool): if True use stemming, else lemmatization
    Returns:
        str: clean preprocessed text
    """
    text = remove_html_tags(text)           # Step 1: Remove HTML
    text = remove_urls(text)                # Step 2: Remove URLs
    text = to_lowercase(text)              # Step 3: Lowercase
    text = remove_punctuation_special(text)# Step 4: Remove punctuation
    tokens = tokenize(text)                # Step 5: Tokenize
    tokens = remove_stopwords(tokens)      # Step 6: Remove stopwords
    if use_stemming:
        tokens = stem_tokens(tokens)       # Step 7a: Stemming
    else:
        tokens = lemmatize_tokens(tokens)  # Step 7b: Lemmatization (default)
    return ' '.join(tokens)                # Rejoin as string

print("✅ Preprocessing functions defined!")

# ── Demonstration on a sample review ─────────────────────────────────────────
sample = df['review'].iloc[0]
print("\n--- BEFORE preprocessing ---")
print(sample[:300])
print("\n--- AFTER preprocessing ---")
print(full_preprocess(sample)[:300])

In [ ]:
# ── Apply preprocessing to the entire dataset ─────────────────────────────────
# Using a sample of 10,000 rows for faster execution (remove [:10000] for full run)
df_sample = df.sample(n=10000, random_state=42).reset_index(drop=True)

print("Preprocessing reviews... (this may take a minute)")
df_sample['clean_review'] = df_sample['review'].apply(full_preprocess)

# Encode target: positive=1, negative=0
df_sample['label'] = df_sample['sentiment'].map({'positive': 1, 'negative': 0})

print(f"✅ Preprocessing complete! {len(df_sample)} reviews processed.")
print("\nSample cleaned review:")
print(df_sample['clean_review'].iloc[0][:300])

In [ ]:
# ── WordCloud Visualization ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, sentiment, color, label in zip(
    axes,
    [1, 0],
    ['Greens', 'Reds'],
    ['Positive Reviews', 'Negative Reviews']
):
    text_corpus = ' '.join(df_sample[df_sample['label'] == sentiment]['clean_review'])
    wc = WordCloud(width=700, height=350, background_color='white',
                   colormap=color, max_words=100).generate(text_corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(label, fontsize=14, fontweight='bold')

plt.suptitle('Most Frequent Words in Reviews', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔢 Step 4: Feature Engineering

We convert the cleaned text into numerical representations using two approaches:

| Method | Description |
|--------|-------------|
| **Bag of Words (BoW)** | Counts word occurrences; ignores order |
| **TF-IDF** | Weights words by importance across documents |

We limit to the top **10,000 features** (words) to keep it efficient.

In [ ]:
# ── Train-Test Split ──────────────────────────────────────────────────────────
X = df_sample['clean_review']
y = df_sample['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")
print(f"Train class distribution:\n{y_train.value_counts()}")

In [ ]:
# ── Bag of Words (BoW) ────────────────────────────────────────────────────────
bow_vectorizer = CountVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_bow = bow_vectorizer.fit_transform(X_train)  # fit on train only
X_test_bow  = bow_vectorizer.transform(X_test)        # transform test

print("BoW Feature Matrix Shape (train):", X_train_bow.shape)

# ── TF-IDF ────────────────────────────────────────────────────────────────────
tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)  # fit on train only
X_test_tfidf  = tfidf_vectorizer.transform(X_test)        # transform test

print("TF-IDF Feature Matrix Shape (train):", X_train_tfidf.shape)
print("\n✅ Feature Engineering complete!")
print("   - BoW captures raw word frequency")
print("   - TF-IDF penalises very common words and rewards rare, meaningful ones")

---
## 🤖 Step 5: Model Building & Training

We train **5 models** on both BoW and TF-IDF features:

1. **Logistic Regression** – Simple, fast, excellent baseline for text
2. **Naive Bayes (Multinomial)** – Probabilistic, works well with BoW/TF-IDF counts
3. **Decision Tree** – Rule-based, interpretable but can overfit
4. **Random Forest** – Ensemble of trees, more robust
5. **XGBoost** – Gradient boosting, powerful ensemble method

In [ ]:
# ── Define Models ─────────────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Naive Bayes'        : MultinomialNB(alpha=1.0),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=15, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost'            : XGBClassifier(n_estimators=100, use_label_encoder=False,
                                          eval_metric='logloss', random_state=42)
}

# ── Helper: Train & Evaluate ──────────────────────────────────────────────────
def evaluate_model(model, X_train, X_test, y_train, y_test):
    """
    Train model and return evaluation metrics dictionary.
    Returns: dict with accuracy, precision, recall, f1
    """
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall'   : round(recall_score(y_test, y_pred), 4),
        'F1 Score' : round(f1_score(y_test, y_pred), 4)
    }, y_pred

# ── Train all models on BoW and TF-IDF ────────────────────────────────────────
results_bow   = {}
results_tfidf = {}
predictions   = {}

print("Training models... Please wait.\n")

for name, model in models.items():
    print(f"  🔹 {name}")

    # Clone-like: re-import to avoid state bleed (create fresh instances)
    import copy

    # BoW
    m_bow = copy.deepcopy(model)
    metrics_bow, preds_bow = evaluate_model(m_bow, X_train_bow, X_test_bow, y_train, y_test)
    results_bow[name] = metrics_bow

    # TF-IDF
    m_tfidf = copy.deepcopy(model)
    metrics_tfidf, preds_tfidf = evaluate_model(m_tfidf, X_train_tfidf, X_test_tfidf, y_train, y_test)
    results_tfidf[name] = metrics_tfidf

    predictions[name] = {'bow': preds_bow, 'tfidf': preds_tfidf}

print("\n✅ All models trained and evaluated!")

---
## 📊 Step 6: Model Evaluation & Comparison

In [ ]:
# ── Results Tables ────────────────────────────────────────────────────────────
df_bow   = pd.DataFrame(results_bow).T
df_tfidf = pd.DataFrame(results_tfidf).T

print("=" * 60)
print("📋 Results: BAG OF WORDS (BoW)")
print("=" * 60)
print(df_bow.to_string())

print("\n" + "=" * 60)
print("📋 Results: TF-IDF")
print("=" * 60)
print(df_tfidf.to_string())

In [ ]:
# ── Grouped Bar Chart: BoW vs TF-IDF Accuracy ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
metrics_to_plot = ['Accuracy', 'F1 Score']
colors = ['#3498db', '#e74c3c']
x = np.arange(len(models))
width = 0.35

for ax, metric in zip(axes, metrics_to_plot):
    bow_vals   = [results_bow[m][metric]   for m in models]
    tfidf_vals = [results_tfidf[m][metric] for m in models]

    bars1 = ax.bar(x - width/2, bow_vals,   width, label='BoW',    color='#3498db', alpha=0.85)
    bars2 = ax.bar(x + width/2, tfidf_vals, width, label='TF-IDF', color='#e74c3c', alpha=0.85)

    ax.set_xlabel('Model', fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f'{metric}: BoW vs TF-IDF', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(list(models.keys()), rotation=20, ha='right', fontsize=9)
    ax.set_ylim(0.5, 1.0)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    # Value labels on bars
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7.5)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7.5)

plt.suptitle('Model Comparison: BoW vs TF-IDF', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap of All Metrics ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, df_res, title in zip(axes,
                              [df_bow, df_tfidf],
                              ['BoW – All Metrics', 'TF-IDF – All Metrics']):
    sns.heatmap(df_res.astype(float), annot=True, fmt='.4f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, vmin=0.5, vmax=1.0,
                annot_kws={'size': 10})
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('')

plt.suptitle('Metrics Heatmap Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrices for Best Model (TF-IDF) ────────────────────────────────
# Determine best model by TF-IDF F1 Score
best_model_name = max(results_tfidf, key=lambda m: results_tfidf[m]['F1 Score'])
print(f"Best model (TF-IDF): {best_model_name}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, vec_type, preds in zip(axes, ['bow', 'tfidf'],
                                [predictions[best_model_name]['bow'],
                                 predictions[best_model_name]['tfidf']]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    ax.set_title(f'{best_model_name}\n({vec_type.upper()})', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle(f'Confusion Matrices – {best_model_name}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Full classification report for best model with TF-IDF
print(f"\nDetailed Classification Report – {best_model_name} (TF-IDF):")
print(classification_report(y_test, predictions[best_model_name]['tfidf'],
                             target_names=['Negative', 'Positive']))

---
## 💡 Step 7: Comparison & Insights

### Best Preprocessing Steps
- **HTML & URL removal** is critical for IMDb data since reviews often contain HTML tags like `<br />`.
- **Lowercasing + punctuation removal** normalises vocabulary and reduces feature space significantly.
- **Stopword removal** removes noise words (e.g., "the", "is") that carry no sentiment signal.
- **Lemmatization** is preferred over stemming here because it produces real words (`running → run`) rather than truncated stems (`running → run` vs `running → runn`), preserving readability and accuracy.

### Best Vectorization Method
- **TF-IDF generally outperforms BoW** for sentiment analysis because it down-weights very common words (e.g., "movie", "film") that appear in both positive and negative reviews equally, and up-weights rare, discriminative words (e.g., "brilliant", "dreadful").
- **Bigrams (ngram_range=(1,2))** help capture phrases like "not good" or "very bad" which would be missed with unigrams alone.

### Best Model & Trade-offs

| Model | Pros | Cons |
|-------|------|------|
| **Logistic Regression** | Fast, accurate, interpretable | Assumes linear separability |
| **Naive Bayes** | Very fast, works with sparse data | Assumes feature independence |
| **Decision Tree** | Interpretable | Prone to overfitting, lower accuracy |
| **Random Forest** | Robust, handles noise well | Slower to train |
| **XGBoost** | Often highest accuracy, handles complex patterns | Slower training, more hyperparameters |

### Conclusion
- **Logistic Regression with TF-IDF** is the best balance of speed and accuracy for text classification.
- **XGBoost with TF-IDF** may achieve the highest F1 score but at the cost of training time.
- For production deployment, **Logistic Regression** would be recommended due to its speed and interpretability.

In [ ]:
# ── Final Summary Table ────────────────────────────────────────────────────────
print("=" * 70)
print("  FINAL SUMMARY – Best F1 Score per Model (TF-IDF)")
print("=" * 70)

summary = []
for name in models:
    bow_f1   = results_bow[name]['F1 Score']
    tfidf_f1 = results_tfidf[name]['F1 Score']
    best_vec = 'TF-IDF' if tfidf_f1 >= bow_f1 else 'BoW'
    best_f1  = max(bow_f1, tfidf_f1)
    summary.append({'Model': name, 'BoW F1': bow_f1,
                    'TF-IDF F1': tfidf_f1, 'Best': best_vec,
                    'Best F1': best_f1})

df_summary = pd.DataFrame(summary).sort_values('Best F1', ascending=False)
df_summary.index = range(1, len(df_summary)+1)
print(df_summary.to_string())

overall_best = df_summary.iloc[0]
print(f"\n🏆 Overall Best Model : {overall_best['Model']}")
print(f"   Best Vectorizer   : {overall_best['Best']}")
print(f"   Best F1 Score     : {overall_best['Best F1']:.4f}")

---
## ✅ End of Assignment

**Summary of Findings:**
- Successfully built a complete NLP pipeline from raw text to trained ML models
- Applied all required preprocessing steps with reusable, modular functions
- Implemented both BoW and TF-IDF feature engineering with bigrams
- Trained and evaluated 5 ML models with Accuracy, Precision, Recall, and F1 Score
- TF-IDF with Logistic Regression delivered the best overall performance
- Visualized results with bar charts, heatmaps, word clouds, and confusion matrices

---
*Submitted as part of Data Science Internship – February 2026 | NLP Assignment 2*